# Robot Data Quality & Success Prediction

真实机器人数据质量审计、固定前缀成功预测，以及独立的仿真清洗实验。

先按 README 在 D 盘完成依赖安装。默认 **inspect** 模式读取已生成的实测结果；改为 **reproduce** 会下载并重跑真实数据流程。没有输出时会明确报错，不生成虚构结果。完整实验、抽样检查与 CI 合成测试的证据层级分别记录。

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
ROOT=Path(os.environ.get('ROBOT_AUDIT_ROOT',Path.cwd())).resolve()
if not (ROOT/'configs/project.yaml').exists() and (ROOT.parent/'configs/project.yaml').exists(): ROOT=ROOT.parent
assert (ROOT/'configs/project.yaml').exists(), 'Run from the project folder'
sys.path.insert(0,str(ROOT/'src'))
from robot_audit.paths import Workspace
ws=Workspace(ROOT);ws.setup()
MODE='inspect'  # change to 'reproduce' for a full real-data rerun
assert MODE in ('inspect','reproduce')
def run(*arguments):
    subprocess.run([sys.executable,'-B','-m','robot_audit','--root',str(ROOT),*arguments],check=True,cwd=ROOT)
print('Workspace:',ROOT)


## 1. 数据与审计
固定数据提交、标签冲突隔离、轨迹去重、结构与转换后时间戳检查。

In [ ]:
if MODE=='reproduce':
    for command in ['fetch','audit','detect']: run(command)
audit=json.loads((ROOT/'reports/audit_summary.json').read_text(encoding='utf-8'))
assert audit['leakage_checks']=='PASS'
print(json.dumps(audit,indent=2,ensure_ascii=False))


## 2. 成功预测
只用前 30 帧运动特征；模型与阈值依验证集选择，测试集按采集者保留。

In [ ]:
if MODE=='reproduce': run('train')
metrics=json.loads((ROOT/'reports/prediction_metrics.json').read_text(encoding='utf-8'))
import pandas as pd
display(pd.DataFrame({name:record['test'] for name,record in metrics['models'].items()}).T)
print('Validation-selected model:',metrics['selected_model'])


## 3. 视频抽样与报告
视频只检查一个明确记录的样本。没有原始传感器时钟，不能据此判断原始时钟漂移。

In [ ]:
if MODE=='reproduce':
    run('video','--episodes','30')
    run('report')
from IPython.display import Image, Markdown, display
import re
report=(ROOT/'reports/research_report.md').read_text(encoding='utf-8')
display(Markdown(re.sub(r'!\[[^\]]*\]\(figures/[^)]+\)','',report)))
for name in ['audit','prediction']:
    display(Image(filename=str(ROOT/f'reports/figures/{name}.png')))
print('REAL-DATA REPORT INSPECTION PASS' if MODE=='inspect' else 'REAL-DATA SEQUENTIAL RUN PASS')


## 4. 策略实验
用 `01_colab_policy.ipynb` 在 Colab 独立运行；达到五组 × 三种子 × 100 次评测并验证初始状态后，才生成 `policy_results.json`。免费配额不足时保留断点，不能将准备完成写成实验完成。

In [ ]:
policy_path=ROOT/'reports/policy_results.json'
if policy_path.exists():
    policy=json.loads(policy_path.read_text(encoding='utf-8'))
    assert policy['status']=='complete' and policy['rollouts']==1500
    display(pd.DataFrame(policy['conditions']).T)
    display(Image(filename=str(ROOT/'reports/figures/policy.png')))
    print('POLICY RESULT INSPECTION PASS')
else:
    print('POLICY EVALUATION PENDING: no improvement claim')
